# PostgreSQL: a hands-on implementation

**Track:** Production · **Guide:** [`059-postgresql.md`](../guides/concepts/059-postgresql.md)

PostgreSQL is a relational database that provides transactions, constraints, indexing, and full-text search.

> **Intuition:** A reliable ledger enforces rules and commits related changes together.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement PostgreSQL]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Persist source documents and an indexing event atomically so a crash cannot silently lose the indexing request.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Design relational tables and constraints

PostgreSQL transactions and constraints keep related application records consistent. This SQLite-backed exercise is runnable locally and uses compatible relational concepts.


In [ ]:
import sqlite3

database = sqlite3.connect(":memory:")
database.execute("PRAGMA foreign_keys = ON")
database.executescript("""
CREATE TABLE documents (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL CHECK(length(title) BETWEEN 1 AND 100),
    body TEXT NOT NULL
);
CREATE TABLE document_tags (
    document_id INTEGER NOT NULL REFERENCES documents(id),
    tag TEXT NOT NULL,
    PRIMARY KEY (document_id, tag)
);
""")

## 2. Commit related writes atomically

A transaction ensures a failed tag write cannot leave a partially saved document.


In [ ]:
try:
    with database:
        cursor = database.execute(
            "INSERT INTO documents(title, body) VALUES (?, ?)",
            ("Password reset", "Reset from account settings."),
        )
        database.execute(
            "INSERT INTO document_tags(document_id, tag) VALUES (?, ?)",
            (cursor.lastrowid, "accounts"),
        )
except sqlite3.DatabaseError as error:
    raise RuntimeError("document transaction failed") from error

print(database.execute("SELECT id, title FROM documents").fetchall())

## 3. Query and inspect the schema-backed result

Use parameterized SQL for values. PostgreSQL adds production features such as `tsvector`, GIN indexes, JSONB, and concurrency controls.


In [ ]:
rows = database.execute(
    """
    SELECT documents.title, document_tags.tag
    FROM documents JOIN document_tags ON documents.id = document_tags.document_id
    WHERE document_tags.tag = ?
    """,
    ("accounts",),
).fetchall()
print(rows)
assert rows == [("Password reset", "accounts")]

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/059-postgresql.md) and the
[learning roadmap](../guides/ROADMAP.md).
